# 02 · Bronze → Silver: Limpeza, Tipagem e Padronização


In [0]:
from pyspark.sql import functions as F 
from pyspark.sql import Window
from pyspark.sql.types import IntegerType

catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## Chamados
**Limpeza dos dados**

### Verificação dos tipos de dados
- Diagnóstico da Base Bruta:
- Todas as colunas vieram em formato texto puro.
- Canal em texto livre com registros inconsistentes misturando 'URA', 'ura', 'U.R.A', 'Chatbot', 'chat bot', 'Atendimento_Inicial', 'Atend. Inicial', 'ATENDIMENTO ESPECIALIZADO'.
- Motivo em texto livre com variações de caixa e acentuação ('CONSULTA DE FATURA', 'Bloqueio de cartao').
- Resolvido com registros inconsistentes misturando 'Sim', 'S', '1', 'Não', 'N', '0', 'Nao resolvido' e texto vazio ("").
- Tempo_espera e tempo_atendimento com o texto 'NULL' no lugar de valores ausentes.
- 2.636 registros duplicados.

In [0]:
df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados")

display(df_bronze_chamados.limit(5))

In [0]:
for c in df_bronze_chamados.columns:
    display(
        df_bronze_chamados.select(
            F.count("*").alias(c),
            F.sum(F.col(c).isNull().cast("int")).alias("nulos"),
            F.sum((F.trim(F.col(c)) == "").cast("int")).alias("vazios")
        )
    )

    display(
        df_bronze_chamados.groupBy(c).count()
                          .orderBy(F.desc("count"))
                          .limit(50)
    )

### Tratamento de valores em 'canais', 'motivos' e 'resolvido'

#### Padronização de Canais:

- 'ura', 'u r a' $\rightarrow$ URA
- 'chatbot', 'chat bot' $\rightarrow$ Chatbot
- 'atendimento inicial', 'atend inicial' $\rightarrow$ Atendimento Inicial
- 'atendimento especializado', 'atend especializado' $\rightarrow$ Atendimento Especializado

In [0]:
df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados")
df_silver_chamados = (
    df_bronze_chamados
        .withColumn("canal", F.lower(F.col("canal")))
        .withColumn("canal", F.regexp_replace(F.col("canal"), r"[^a-z]+", " "))
        .withColumn("canal", F.trim(F.col("canal")))
        .withColumn("canal",
            F.when(F.col("canal").isin("ura", "u r a"), "URA")
            .when(F.col("canal").isin("chatbot", "chat bot"), "Chatbot")
            .when(F.col("canal").isin("atendimento inicial", "atend inicial"), "Atendimento Inicial")
            .when(F.col("canal").isin("atendimento especializado", "atend especializado"), "Atendimento Especializado")
            .otherwise("Não Informado")
    )
)

display(df_silver_chamados.groupBy("canal").count().orderBy(F.desc("count")))


#### Padronização de Motivos: 
lower, remoção de acentos (unicodedata), remoção de caracteres especiais e trim, harmonizando para os 14 motivos oficiais do case (ex.: 'bloqueio de cartao' $\rightarrow$ Bloqueio de cartão).

In [0]:
import unicodedata
from pyspark.sql.types import StringType

@F.udf(StringType())
def remover_acentos(texto):
    if texto is None:
        return None
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

df_silver_chamados = (
    df_silver_chamados
        .withColumn("motivo", F.lower(F.col("motivo")))
        .withColumn("motivo", remover_acentos(F.col("motivo")))
        .withColumn("motivo", F.regexp_replace(F.col("motivo"), r"[^a-z]+", " "))
        .withColumn("motivo", F.trim(F.col("motivo")))
        .withColumn("motivo",
            F.when(F.col("motivo") == "consulta de fatura", "Consulta de fatura")
            .when(F.col("motivo") == "consulta de limite", "Consulta de limite")
            .when(F.col("motivo") == "consulta de contrato", "Consulta de contrato")
            .when(F.col("motivo") == "contestacao de fatura", "Contestação de fatura")
            .when(F.col("motivo") == "renegociacao de divida", "Renegociação de dívida")
            .when(F.col("motivo") == "bloqueio de cartao", "Bloqueio de cartão")
            .when(F.col("motivo") == "desbloqueio de cartao", "Desbloqueio de cartão")
            .when(F.col("motivo") == "contratacao de cartao adicional", "Contratação de cartão adicional")
            .when(F.col("motivo") == "compra nao autorizada", "Compra não autorizada")
            .when(F.col("motivo") == "alteracao de dados cadastrais vencimento da fatura telefone email", "Alteração de dados cadastrais (vencimento da fatura, telefone, email)")
            .when(F.col("motivo") == "transferencia de agencia", "Transferência de agência")
            .when(F.col("motivo") == "cancelamento de conta", "Cancelamento de conta")
            .when(F.col("motivo") == "problema com aplicativo", "Problema com aplicativo")
            .when(F.col("motivo") == "duvidas gerais sobre programa de pontos", "Dúvidas gerais sobre programa de pontos")
            .otherwise("Não Informado")
        )
)

display(df_silver_chamados.groupBy("motivo").count().orderBy(F.desc("count")))


#### Padronização de Resolvido:
- 'sim', 's', '1' -> Sim
- 'não', 'n', '0', 'nao resolvido' -> Não
- Vazio/nulo -> Não Informado

In [0]:
df_silver_chamados = (
    df_silver_chamados
        .withColumn("resolvido", F.lower(F.col("resolvido")))
        .withColumn("resolvido", F.trim(F.col("resolvido")))
        .withColumn("resolvido",
            F.when(F.col("resolvido").isin("não", "n", "0", "nao resolvido"), "Não")
            .when(F.col("resolvido").isin("sim", "s", "1"), "Sim")
            .otherwise("Não Informado")
    )
)

display(df_silver_chamados.groupBy("resolvido").count().orderBy(F.desc("count")))


### Tipagem:
tempo_espera, tempo_atendimento e id_atendente convertidos para IntegerType() com try_cast.

In [0]:
df_silver_chamados = (
    df_silver_chamados
        .withColumn("tempo_espera", F.expr("try_cast(tempo_espera as int)"))
        .withColumn("tempo_atendimento", F.expr("try_cast(tempo_atendimento as int)"))
        .withColumn("id_atendente", F.expr("try_cast(id_atendente as int)"))
)

df_silver_chamados.printSchema()

### Linhas duplicadas

#### Verificar dados duplicados

In [0]:
print("linhas:", df_bronze_chamados.count())
print("ids distintos:", df_bronze_chamados.select("id_chamado", "id_cliente", "canal", "motivo", "resolvido").distinct().count())
print("ids duplicados:", df_bronze_chamados.count() - df_bronze_chamados.select("id_chamado", "id_cliente", "canal", "motivo", "resolvido").distinct().count())

#### Remover valores duplicados

dropDuplicates por id_chamado, id_cliente, canal, motivo e resolvido.

In [0]:
print("linhas antes:", df_silver_chamados.count())

df_silver_chamados = (
    df_silver_chamados
    .dropDuplicates(["id_chamado", "id_cliente", "canal", "motivo", "resolvido"])
)

print("linhas depois:", df_silver_chamados.count())

### Gravação do tb_chamados

In [0]:
df_silver_chamados.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.tb_chamados")

display(df_bronze_chamados.limit(10))
display(df_silver_chamados.limit(10))


## Atendentes e motivos
##A primeira vista, é possível notar algumas inconsistências nos dados, como:
- **Nos atendentes:** tem nível misturando `'1'`, `'Nível 1'`, `'2'`, `'N2'`; datas em formato de texto (`dd/MM/yyyy`); e campos vazios quando a pessoa ainda não foi desligada.
- **Nos motivos:** tem categorias que vieram em branco (`""`), algumas escritas sem acento (`Cartao`, `Beneficios`), e criticidade toda misturada (`alta`, `Media`, `baixa`).

---
## 1. Tratamento dos Atendentes (`silver.tb_atendentes`)

O que está sendo feito:
- **`id_atendente`**: passa de string pra número inteiro (`IntegerType`);
- **`nivel_atendimento`**: unifica tudo em **`Nível 1`** ou **`Nível 2`** (troca '1' e 'N1' por 'Nível 1', e '2' e 'N2' por 'Nível 2');
- **`data_admissao`**: converte o texto brasileiro `dd/MM/yyyy` pra data de verdade (`DateType`);
- **`data_desligamento`**: quem tá com string vazia vira `NULL` (quem ainda tá ativo na empresa), e quem tem data vira `DateType`;
- **`status_atendente`**: padroniza em caixa alta inicial (`Ativo` / `Inativo`).


In [0]:
df_bronze_atendentes = spark.table(f"{bronze_schema}.tb_atendentes")

win_atendente = Window.partitionBy("id_atendente").orderBy(F.col("ingestion_datetime").desc())

df_silver_atendentes = (
    df_bronze_atendentes
    .withColumn("rn", F.row_number().over(win_atendente))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        F.col("id_atendente").cast(IntegerType()).alias("id_atendente"),
        
        F.trim(F.col("nome_atendente")).alias("nome_atendente"),
        
        F.when(F.col("nivel_atendimento").isin("1", "Nível 1", "Nivel 1", "N1"), "Nível 1")
         .when(F.col("nivel_atendimento").isin("2", "Nível 2", "Nivel 2", "N2"), "Nível 2")
         .otherwise("Não Informado")
         .alias("nivel_atendimento"),
        
        F.to_date(F.col("data_admissao"), "dd/MM/yyyy").alias("data_admissao"),
        
        F.when(
            (F.col("data_desligamento").isNull()) | (F.trim(F.col("data_desligamento")) == ""), 
            F.lit(None)
        ).otherwise(
            F.to_date(F.col("data_desligamento"), "dd/MM/yyyy")
        ).alias("data_desligamento"),
        
        F.initcap(F.trim(F.col("status_atendente"))).alias("status_atendente"),
        
        F.col("ingestion_datetime")
    )
)

(
    df_silver_atendentes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_atendentes")
)

print(f"Tabela {silver_schema}.tb_atendentes salva com sucesso!")
print(f"Total de atendentes: {df_silver_atendentes.count()}")

In [0]:
print("Contagem por Nível e Status:")
df_silver_atendentes.groupBy("nivel_atendimento", "status_atendente").count().show()

display(df_silver_atendentes.limit(5))

---
## 2. Tratamento dos Motivos (`silver.dim_motivo`)

O que está sendo feito:
- **`id_motivo`**: vira número inteiro (`IntegerType`);
- **Preenchimento de categorias em branco**: no arquivo original vieram 3 motivos sem categoria. Esses foram os critérios usados para classificação:
  - *'Transferência de agência'* $\rightarrow$ **Cadastral** (dados cadastrais do cliente/agência);
  - *'Contratação de cartão adicional'* $\rightarrow$ **Cartão** (operações e contratações do cartão);
  - *'Cancelamento de conta'* $\rightarrow$ **Financeiro** (contrato/encerramento da conta com o banco);
- **Acentuação**: arruma `'Cartao'` pra `'Cartão'` e `'Beneficios'` pra `'Benefícios'`;
- **Criticidade**: padroniza palavras com mistura de maiúsculas/minúsculas pra **`Alta`**, **`Média`** ou **`Baixa`**.

In [0]:
df_bronze_motivos = spark.table(f"{bronze_schema}.tb_motivos")

win_motivo = Window.partitionBy("id_motivo").orderBy(F.col("ingestion_datetime").desc())

df_silver_motivos = (
    df_bronze_motivos
    .withColumn("rn", F.row_number().over(win_motivo))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        F.col("id_motivo").cast(IntegerType()).alias("id_motivo"),
        
        F.trim(F.col("nome_motivo")).alias("nome_motivo"),
        
        F.when(F.col("nome_motivo") == "Transferência de agência", "Cadastral")
         .when(F.col("nome_motivo") == "Contratação de cartão adicional", "Cartão")
         .when(F.col("nome_motivo") == "Cancelamento de conta", "Financeiro")
         .when(F.lower(F.col("categoria")).isin("cartao", "cartão"), "Cartão")
         .when(F.lower(F.col("categoria")).isin("beneficios", "benefícios"), "Benefícios")
         .when(F.lower(F.col("categoria")) == "financeiro", "Financeiro")
         .when(F.lower(F.col("categoria")) == "cadastral", "Cadastral")
         .when(F.lower(F.col("categoria")) == "atendimento", "Atendimento")
         .otherwise(F.initcap(F.trim(F.col("categoria"))))
         .alias("categoria"),
        
        F.when(F.lower(F.col("criticidade")).isin("alta"), "Alta")
         .when(F.lower(F.col("criticidade")).isin("media", "média"), "Média")
         .when(F.lower(F.col("criticidade")).isin("baixa"), "Baixa")
         .otherwise(F.initcap(F.trim(F.col("criticidade"))))
         .alias("criticidade"),
        
        F.col("ingestion_datetime")
    )
)

(
    df_silver_motivos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.tb_motivos")
)

print(f"Tabela {silver_schema}.tb_motivos salva com sucesso!")
print(f"Total de motivos: {df_silver_motivos.count()}")


In [0]:
print("Resumo das categorias e criticidades tratadas:")
df_silver_motivos.groupBy("categoria", "criticidade").count().orderBy("categoria").show()

display(df_silver_motivos.orderBy("id_motivo"))


Assim, as tabelas `silver.tb_atendentes` e `silver.tb_motivos` estão prontas, higienizadas e padronizadas.
Todos os valores em branco das categorias foram tratados, os tipos estão ajustados corretamente e o pipeline pode rodar livremente sem travamento ou duplicação de dados.


In [0]:
catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## Clientes

### Transformações Aplicadas

- Remoção de registros com `id_cliente` nulo.
- Padronização da coluna `nome`, removendo espaços extras e formatando em estilo título.
- Padronização da coluna `email`, convertendo os valores para letras minúsculas e removendo espaços desnecessários.
- Padronização da coluna `regiao`, garantindo nomenclatura consistente.
- Conversão da coluna `data_cadastro` para o tipo `Date`, tratando os formatos encontrados na origem.
- Criação da coluna `ano_cadastro` a partir da data de cadastro.
- Validação da coluna `idade`, mantendo apenas valores entre 0 e 120 anos.
- Validação da coluna `dia_vencimento_fatura`, mantendo apenas valores entre 1 e 31.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

df_bronze_clientes = spark.table(
    f"{bronze_schema}.tb_clientes"
)

window_clientes = (
    Window
    .partitionBy("id_cliente")
    .orderBy(
        F.col("ingestion_datetime").desc()
    )
)

tb_clientes = (
    df_bronze_clientes

    .filter(
        F.col("id_cliente").isNotNull()
    )

    .withColumn(
    "row_num",
    F.row_number().over(window_clientes)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")

    .withColumn(
        "nome",
        F.initcap(
            F.trim(F.col("nome"))
        )
    )

    .withColumn(
        "email",
        F.lower(
            F.trim(F.col("email"))
        )
    )

    .withColumn(
        "regiao",
        F.initcap(
            F.trim(F.col("regiao"))
        )
    )

    .withColumn(
        "data_cadastro",
        F.when(
            F.col("data_cadastro").rlike("^\\d{4}-\\d{2}-\\d{2}$"),
            F.to_date(
                F.col("data_cadastro"),
                "yyyy-MM-dd"
            )
        )
        .when(
            F.col("data_cadastro").rlike("^\\d{2}/\\d{2}/\\d{4}$"),
            F.to_date(
                F.col("data_cadastro"),
                "dd/MM/yyyy"
            )
        )
    )

    .withColumn(
        "ano_cadastro",
        F.year("data_cadastro")
    )

    .withColumn(
        "idade",
        F.when(
            (F.col("idade") >= 0) &
            (F.col("idade") <= 120),
            F.col("idade")
        )
    )

    .withColumn(
        "dia_vencimento_fatura",
        F.when(
            (F.col("dia_vencimento_fatura") >= 1) &
            (F.col("dia_vencimento_fatura") <= 31),
            F.col("dia_vencimento_fatura")
        )
    )

    .select(
        "id_cliente",
        "nome",
        "email",
        "regiao",
        "idade",
        "data_cadastro",
        "ano_cadastro",
        "dia_vencimento_fatura"
    )
)

tb_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{silver_schema}.tb_clientes"
    )

display(
    spark.table(
        f"{silver_schema}.tb_clientes"
    ).limit(10)
)

### Validações de Qualidade

Após a carga, foram realizadas verificações para:

- Quantidade total de registros.
- Registros com `id_cliente` nulo.
- Registros com `nome` nulo.
- Registros com `email` nulo.
- Existência de registros duplicados por `id_cliente`.

In [0]:
display(
    tb_clientes.select(
        F.count("*").alias("total_registros"),
        F.sum(
            F.col("id_cliente").isNull().cast("int")
        ).alias("id_cliente_nulos"),
        F.sum(
            F.col("nome").isNull().cast("int")
        ).alias("nome_nulos"),
        F.sum(
            F.col("email").isNull().cast("int")
        ).alias("email_nulos")
    )
)

## Chamados_Hora
## 02 · Bronze → Silver: Limpeza, Tipagem e Correção de Datas

A tabela `vcredit.silver.tb_chamados_hora` é a versão limpa, tipada e padronizada dos chamados da camada Bronze, preparada para alimentar a camada Gold. Os problemas de qualidade são corrigidos apenas quando há segurança para isso ou sinalizados em colunas `dq_...`, sem exclusão de linhas; somente IDs inválidos e duplicados são desviados para uma tabela de quarentena.

### Inconsistências identificadas na Bronze:
- **Horários:** as 3 colunas de data estão armazenadas como texto (`dd/MM/yyyy às HH:mm:ss`);
- **Aberturas:** 198 chamados com abertura em 2027, mas com início e finalização em 2025/2026;
- **Inícios:** 3.414 chamados sem hora de início;
- **Finalização:** 1 chamado com finalização anterior ao início.

### Objetivo desta etapa:
- Consumir a tabela `vcredit.bronze.tb_chamados_hora`, aplicando limpeza básica de texto.
- Isolar em quarentena os registros com `id_chamado` inválido ou duplicado.
- Converter `id_chamado` para inteiro e os horários para `TimestampType`.
- Corrigir o ano da abertura quando houver divergência evidente em relação ao início e à finalização.
- Calcular os tempos de espera e de atendimento e classificar a situação de cada chamado.
- Sinalizar os problemas de qualidade por meio de colunas `dq_...` e de um status consolidado.
- Garantir idempotência e integridade na gravação em formato **Delta Lake**.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

print(f"Lendo da Bronze: {bronze_schema}")
print(f"Gravando na Silver: {silver_schema}")

---
## Parâmetros de Configuração

### Critérios ajustáveis:
Os valores abaixo concentram os critérios utilizados nas regras de tratamento e podem ser alterados sem modificar o restante do código:
- **`PERIODO_INICIO`** e **`PERIODO_FIM`**: período esperado para a abertura dos chamados;
- **`CORRIGIR_ANO_ABERTURA`**: habilita ou desabilita a correção do ano da abertura;
- **`LIMIAR_DIAS_ANO_DIVERGENTE`** e **`MAX_DURACAO_CORRECAO_SEG`**: condições para aceitar a correção (abertura posterior em 180 dias ou mais e duração resultante de até 24 horas);
- **`MAX_ESPERA_SEG`** e **`MAX_ATENDIMENTO_SEG`**: limites para sinalização de outliers (apenas marcação, sem alteração dos valores);
- **`FORMATOS_DATA`**: formatos de data e hora aceitos na conversão.

In [0]:
PERIODO_INICIO = "2025-01-01 00:00:00"
PERIODO_FIM = "2026-07-31 23:59:59"

CORRIGIR_ANO_ABERTURA = True  # True = corrige o ano da abertura
LIMIAR_DIAS_ANO_DIVERGENTE = 180  # abertura 180+ dias depois do início/fim = erro de ano
MAX_DURACAO_CORRECAO_SEG = 86400  # depois de corrigir, o chamado dura até 24h

MAX_ESPERA_SEG = 3600  # espera acima de 1h = outlier (só marca)
MAX_ATENDIMENTO_SEG = 7200  # atendimento acima de 2h = outlier (só marca)

FORMATOS_DATA = [
    "d/M/yyyy H:mm:ss",
    "d/M/yyyy H:mm",
    "yyyy-M-d H:mm:ss",
    "yyyy-M-d H:mm",
    "yyyy-MM-dd'T'HH:mm:ss",
]

---
## 1. Leitura da Bronze (`vcredit.bronze.tb_chamados_hora`)

A Bronze é lida integralmente em formato texto, e as seguintes padronizações são aplicadas:
- Remoção de espaços em branco (`trim`) em todas as colunas;
- Conversão de textos vazios em `NULL`;
- Preservação dos valores originais em colunas com sufixo `_raw`, mantendo também `ingestion_datetime`.

In [0]:
df_bronze = spark.table(f"{bronze_schema}.tb_chamados_hora")
total_bronze = df_bronze.count()

def limpa_texto(coluna):
    texto = F.trim(F.col(coluna))
    return F.when(texto == "", F.lit(None)).otherwise(texto)

df_base = df_bronze.select(
    limpa_texto("ID_Chamado").alias("id_chamado_raw"),
    limpa_texto("ID_Cliente").alias("id_cliente_raw"),
    limpa_texto("Hora_Abertura_Chamado").alias("abertura_raw"),
    limpa_texto("Hora_Inicio_Atendimento").alias("inicio_raw"),
    limpa_texto("Hora_Finalizacao_Atendimento").alias("fim_raw"),
    F.col("ingestion_datetime")
)

print(f"Linhas na Bronze: {total_bronze}")

---
## 2. IDs inválidos e duplicados

A unicidade de `id_chamado` é assegurada antes de qualquer conversão de tipo:
- **`id_chamado` ausente ou não numérico**: o registro é encaminhado à quarentena com o motivo `ID_CHAMADO_AUSENTE_OU_INVALIDO`;
- **`id_chamado` duplicado**: permanece a linha mais completa (menor quantidade de campos vazios), com desempate pela ingestão mais recente e, em seguida, pelo `hash_linha`. As demais seguem para a quarentena com o motivo `ID_CHAMADO_DUPLICADO_DESCARTADO`;
- **`qtd_linhas_bronze`**: registra quantas linhas existiam na Bronze para cada ID e alimenta a marcação `dq_id_duplicado`.

In [0]:
colunas_raw = ["id_chamado_raw", "id_cliente_raw", "abertura_raw", "inicio_raw", "fim_raw"]

df_base = (
    df_base
    .withColumn("id_chamado_txt", F.regexp_replace(F.col("id_chamado_raw"), r"\.0+$", ""))
    .withColumn("id_valido", F.coalesce(F.col("id_chamado_txt").rlike(r"^\d{1,9}$"), F.lit(False)))
    .withColumn("hash_linha", F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("")) for c in colunas_raw]), 256))
)

df_invalidos = df_base.filter(~F.col("id_valido")).withColumn("motivo", F.lit("ID_CHAMADO_AUSENTE_OU_INVALIDO"))

campos_vazios = sum([F.when(F.col(c).isNull(), 1).otherwise(0) for c in colunas_raw[1:]])

win_chamado = Window.partitionBy("id_chamado_txt").orderBy(
    campos_vazios.asc(), F.col("ingestion_datetime").desc(), F.col("hash_linha").asc()
)
win_contagem = Window.partitionBy("id_chamado_txt")

df_validos = (
    df_base.filter(F.col("id_valido"))
    .withColumn("rn", F.row_number().over(win_chamado))
    .withColumn("qtd_linhas_bronze", F.count("*").over(win_contagem))
)

df_duplicados = df_validos.filter(F.col("rn") > 1).withColumn("motivo", F.lit("ID_CHAMADO_DUPLICADO_DESCARTADO"))
df_unicos = df_validos.filter(F.col("rn") == 1).drop("rn")

def formata_quarentena(df):
    return df.select(
        F.col("id_chamado_raw").alias("id_chamado"),
        F.col("id_cliente_raw").alias("id_cliente"),
        F.col("abertura_raw").alias("hora_abertura_chamado"),
        F.col("inicio_raw").alias("hora_inicio_atendimento"),
        F.col("fim_raw").alias("hora_finalizacao_atendimento"),
        F.col("motivo"),
        F.col("ingestion_datetime")
    )

df_quarentena = formata_quarentena(df_invalidos).unionByName(formata_quarentena(df_duplicados))

---
## 3. Tipagem

- **`id_chamado`**: conversão de `string` para inteiro (`IntegerType`);
- **`id_cliente`**: permanece como texto, com remoção do sufixo `.0` e de separadores (`.`, `-`, `/` e espaços), sendo mantido apenas quando o resultado for numérico;
- **Horários**: convertidos para `TimestampType` a partir dos formatos definidos em `FORMATOS_DATA`, desconsiderando o texto intermediário (ex.: "às") entre data e hora;
- **`veio_abertura`**, **`veio_inicio`** e **`veio_fim`**: indicam se o valor original estava preenchido, permitindo identificar horários ilegíveis.

In [0]:
def converte_data(coluna):
    texto = F.regexp_replace(
        F.col(coluna),
        r"^(\d{1,2}/\d{1,2}/\d{4})\D+(\d{1,2}:\d{2}(:\d{2})?)$",
        "$1 $2"
    )
    return F.coalesce(*[F.try_to_timestamp(texto, F.lit(formato)) for formato in FORMATOS_DATA])

cliente_limpo = F.regexp_replace(F.regexp_replace(F.col("id_cliente_raw"), r"\.0+$", ""), r"[\.\-\s/]", "")

df_tipado = df_unicos.select(
    F.col("id_chamado_txt").cast(IntegerType()).alias("id_chamado"),
    F.when(cliente_limpo.rlike(r"^\d+$"), cliente_limpo).alias("id_cliente"),
    converte_data("abertura_raw").alias("hora_abertura_original"),
    converte_data("inicio_raw").alias("hora_inicio_atendimento"),
    converte_data("fim_raw").alias("hora_finalizacao_atendimento"),
    F.col("abertura_raw").isNotNull().alias("veio_abertura"),
    F.col("inicio_raw").isNotNull().alias("veio_inicio"),
    F.col("fim_raw").isNotNull().alias("veio_fim"),
    F.col("qtd_linhas_bronze"),
    F.col("ingestion_datetime")
)

---
## 4. Correção do ano da abertura

- Este é o **único** horário alterado na camada Silver: a abertura com ano divergente (ex.: 2027) é ajustada para o ano do início do atendimento ou, na ausência deste, da finalização;
- A correção ocorre somente quando a abertura é posterior ao início/finalização em 180 dias ou mais e o chamado resultante tem duração de até 24 horas;
- O valor original é preservado em **`hora_abertura_original`**, e a coluna **`corr_ano_abertura`** indica os registros corrigidos.

In [0]:
df_ano = (
    df_tipado
    .withColumn("ref_ts", F.coalesce("hora_inicio_atendimento", "hora_finalizacao_atendimento"))
    .withColumn("fim_ref", F.coalesce("hora_finalizacao_atendimento", "hora_inicio_atendimento"))
    .withColumn("dif_anos", F.year("ref_ts") - F.year("hora_abertura_original"))
    .withColumn("abertura_try1", F.expr("hora_abertura_original + make_interval(dif_anos, 0, 0, 0, 0, 0, 0)"))
    .withColumn(
        "abertura_ajustada",
        F.when(
            F.col("abertura_try1") > F.col("ref_ts"),
            F.expr("hora_abertura_original + make_interval(dif_anos - 1, 0, 0, 0, 0, 0, 0)")
        ).otherwise(F.col("abertura_try1"))
    )
    .withColumn(
        "corr_ano_abertura",
        F.lit(CORRIGIR_ANO_ABERTURA) & F.coalesce(
            (F.datediff("hora_abertura_original", "ref_ts") >= LIMIAR_DIAS_ANO_DIVERGENTE)
            & (F.col("fim_ref").cast("long") - F.col("abertura_ajustada").cast("long")).between(0, MAX_DURACAO_CORRECAO_SEG),
            F.lit(False)
        )
    )
    .withColumn(
        "hora_abertura_chamado",
        F.when(F.col("corr_ano_abertura"), F.col("abertura_ajustada")).otherwise(F.col("hora_abertura_original"))
    )
)

---
## 5. Tempos e situação do atendimento

- **`espera_seg`** (início − abertura), **`atendimento_seg`** (finalização − início) e **`duracao_total_seg`** (finalização − abertura);
- **`situacao_atendimento`**: `ATENDIDO`, `NAO_ATENDIDO` (sem início, indicando provável abandono), `SEM_FINALIZACAO` ou `SEM_HORARIOS`;
- **`tempo_ate_desistencia_seg`**: duração total dos chamados `NAO_ATENDIDO`, quando maior ou igual a zero;
- **`espera_valida`** e **`atendimento_valido`**: indicam tempo maior ou igual a zero, devendo ser utilizadas no cálculo de médias;
- **`data_abertura`**, **`hora_abertura`** e **`ano_mes`**: atributos derivados do horário de abertura.

In [0]:
df_calc = (
    df_ano
    .withColumn("espera_seg", F.col("hora_inicio_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
    .withColumn("atendimento_seg", F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_inicio_atendimento").cast("long"))
    .withColumn("duracao_total_seg", F.col("hora_finalizacao_atendimento").cast("long") - F.col("hora_abertura_chamado").cast("long"))
    .withColumn(
        "situacao_atendimento",
        F.when(F.col("hora_inicio_atendimento").isNotNull() & F.col("hora_finalizacao_atendimento").isNotNull(), "ATENDIDO")
         .when(F.col("hora_inicio_atendimento").isNull() & F.col("hora_finalizacao_atendimento").isNotNull(), "NAO_ATENDIDO")
         .when(F.col("hora_inicio_atendimento").isNotNull(), "SEM_FINALIZACAO")
         .otherwise("SEM_HORARIOS")
    )
    .withColumn(
        "tempo_ate_desistencia_seg",
        F.when((F.col("situacao_atendimento") == "NAO_ATENDIDO") & (F.col("duracao_total_seg") >= 0), F.col("duracao_total_seg"))
    )
    .withColumn("espera_valida", F.coalesce(F.col("espera_seg") >= 0, F.lit(False)))
    .withColumn("atendimento_valido", F.coalesce(F.col("atendimento_seg") >= 0, F.lit(False)))
    .withColumn("data_abertura", F.to_date("hora_abertura_chamado"))
    .withColumn("hora_abertura", F.hour("hora_abertura_chamado"))
    .withColumn("ano_mes", F.date_format("hora_abertura_chamado", "yyyy-MM"))
)

---
## 6. Marcações de qualidade

- Cada problema identificado é registrado em uma coluna `dq_...` (`true`/`false`), sem exclusão de linhas;
- Finalização anterior ao início **não é alterada**, apenas sinalizada;
- Valores de espera e de atendimento acima dos limites configurados são marcados como outliers, sem alteração do dado;
- **`dq_status`**: classificação consolidada, aplicando-se a primeira condição verdadeira:
  - `INCONSISTENTE`: horário ilegível, abertura nula, ordem cronológica inválida ou abertura fora do período;
  - `INCOMPLETO`: ausência de início, de finalização ou de `id_cliente`;
  - `CORRIGIDO`: ano da abertura ajustado;
  - `OK`: nenhuma das condições anteriores.

In [0]:
df_flags = (
    df_calc
    .withColumn("dq_id_duplicado", F.col("qtd_linhas_bronze") > 1)
    .withColumn("dq_id_cliente_nulo", F.col("id_cliente").isNull())
    .withColumn("dq_abertura_nula", F.col("hora_abertura_chamado").isNull())
    .withColumn("dq_inicio_nulo", F.col("hora_inicio_atendimento").isNull())
    .withColumn("dq_fim_nulo", F.col("hora_finalizacao_atendimento").isNull())
    .withColumn(
        "dq_ts_ilegivel",
        (F.col("veio_abertura") & F.col("hora_abertura_original").isNull())
        | (F.col("veio_inicio") & F.col("hora_inicio_atendimento").isNull())
        | (F.col("veio_fim") & F.col("hora_finalizacao_atendimento").isNull())
    )
    .withColumn("dq_inicio_antes_abertura", F.coalesce(F.col("espera_seg") < 0, F.lit(False)))
    .withColumn("dq_fim_antes_inicio", F.coalesce(F.col("atendimento_seg") < 0, F.lit(False)))
    .withColumn("dq_fim_antes_abertura", F.coalesce(F.col("duracao_total_seg") < 0, F.lit(False)))
    .withColumn(
        "dq_fora_periodo",
        F.coalesce(
            (F.col("hora_abertura_chamado") < F.lit(PERIODO_INICIO).cast("timestamp"))
            | (F.col("hora_abertura_chamado") > F.lit(PERIODO_FIM).cast("timestamp")),
            F.lit(False)
        )
    )
    .withColumn("dq_espera_outlier", F.coalesce(F.col("espera_seg") > MAX_ESPERA_SEG, F.lit(False)))
    .withColumn("dq_atendimento_outlier", F.coalesce(F.col("atendimento_seg") > MAX_ATENDIMENTO_SEG, F.lit(False)))
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_ts_ilegivel") | F.col("dq_abertura_nula") | F.col("dq_inicio_antes_abertura")
            | F.col("dq_fim_antes_inicio") | F.col("dq_fim_antes_abertura") | F.col("dq_fora_periodo"),
            "INCONSISTENTE"
        )
         .when(F.col("dq_inicio_nulo") | F.col("dq_fim_nulo") | F.col("dq_id_cliente_nulo"), "INCOMPLETO")
         .when(F.col("corr_ano_abertura"), "CORRIGIDO")
         .otherwise("OK")
    )
)

---
## 7. Gravação (`vcredit.silver.tb_chamados_hora` e `vcredit.silver.tb_chamados_hora_quarentena`)

- Seleção das colunas finais da Silver, com os horários, tempos, situação e marcações de qualidade;
- Gravação da tabela principal e da quarentena em formato Delta, com `overwrite` e `overwriteSchema`, de modo que a reexecução não gere duplicidade de registros.

In [0]:
df_silver_chamados_hora = df_flags.select(
    "id_chamado",
    "id_cliente",
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento",
    "hora_abertura_original",
    "espera_seg",
    "atendimento_seg",
    "duracao_total_seg",
    "tempo_ate_desistencia_seg",
    "situacao_atendimento",
    "espera_valida",
    "atendimento_valido",
    "data_abertura",
    "hora_abertura",
    "ano_mes",
    "corr_ano_abertura",
    "dq_id_duplicado",
    "dq_id_cliente_nulo",
    "dq_abertura_nula",
    "dq_inicio_nulo",
    "dq_fim_nulo",
    "dq_ts_ilegivel",
    "dq_inicio_antes_abertura",
    "dq_fim_antes_inicio",
    "dq_fim_antes_abertura",
    "dq_fora_periodo",
    "dq_espera_outlier",
    "dq_atendimento_outlier",
    "dq_status",
    "ingestion_datetime"
)

(
    df_silver_chamados_hora.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_chamados_hora")
)

(
    df_quarentena.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_chamados_hora_quarentena")
)

print(f"Tabela {silver_schema}.tb_chamados_hora salva com sucesso!")
print(f"Tabela {silver_schema}.tb_chamados_hora_quarentena salva com sucesso!")
print(f"Total de chamados na Silver: {df_silver_chamados_hora.count()}")
print(f"Total de registros em quarentena: {df_quarentena.count()}")
display(df_silver_chamados_hora)


---
Dessa forma, a tabela `vcredit.silver.tb_chamados_hora` fica tipada e com os problemas tratados ou sinalizados, pronta para consumo pela camada Gold. Os registros desviados constam em `vcredit.silver.tb_chamados_hora_quarentena`.

## Canais, Pesquisa_satisfação e Custos
- `pesquisa_satisfacao.csv`
- `custos.csv`
- `canais.csv`

### 2. Limpeza dos dados

#### 2.1. Pesquisa de satisfação
**Tratamentos realizados: remoção de espaços, conversão da nota para inteiro, validação das notas entre 1 e 5 e identificação de duplicidades.**

In [0]:
df_pesquisa_bronze = spark.table(f"{bronze_schema}.tb_pesquisa_satisfacao")
df_pesquisa = (
    df_pesquisa_bronze
    .withColumn(
        "id_pesquisa",
        F.when(
            F.trim(F.col("id_pesquisa").cast("string")) == "",
            None
        ).otherwise(F.trim(F.col("id_pesquisa").cast("string")))
    )
    .withColumn(
        "id_chamado",
        F.when(
            F.trim(F.col("id_chamado").cast("string")) == "",
            None
        ).otherwise(F.trim(F.col("id_chamado").cast("string")))
    )
    .withColumn(
        "nota_atendimento",
        F.expr("try_cast(NULLIF(TRIM(nota_atendimento), 'NULL') AS INT)")
    )
    .withColumn(
        "nota_valida",
        F.col("nota_atendimento").between(1, 5)
    )
    .withColumn(
        "id_pesquisa_duplicado",
        F.col("id_pesquisa").isNotNull() &
        (F.count("id_pesquisa").over(
            __import__("pyspark").sql.Window.partitionBy("id_pesquisa")
        ) > 1)
    )
)

df_pesquisa = df_pesquisa.dropDuplicates()
(
    df_pesquisa.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.pesquisa_satisfacao")
)

display(df_pesquisa.limit(10))

### 2.2. Custos
**Tratamentos realizados: remoção de símbolos monetários e textos desnecessários, padronização dos separadores decimais, conversão para DECIMAL(12,2), preservação do valor original e validação de custos não negativos.**

In [0]:
df_custos_bronze = spark.table(f"{bronze_schema}.tb_custos")
df_custos = (
    df_custos_bronze
    .withColumn(
        "id_custo",
        F.trim(F.col("id_custo").cast("string"))
    )
    .withColumn(
        "id_chamado",
        F.trim(F.col("id_chamado").cast("string"))
    )
    .withColumn(
        "custo_original",
        F.col("custo").cast("string")
    )
)

valor_limpo = F.lower(F.trim(F.col("custo_original")))
valor_limpo = F.regexp_replace(valor_limpo, r"r\$", "")
valor_limpo = F.regexp_replace(valor_limpo, r"reais?", "")
valor_limpo = F.regexp_replace(valor_limpo, r"\s+", "")
valor_limpo = F.regexp_replace(valor_limpo, r"[^0-9,.\-]", "")

tem_virgula = F.instr(valor_limpo, ",") > 0
tem_ponto = F.instr(valor_limpo, ".") > 0
virgula_depois_ponto = (F.instr(valor_limpo, ",") > F.instr(valor_limpo, "."))

valor_normalizado = (
    F.when(
        tem_virgula & tem_ponto & virgula_depois_ponto,
        F.regexp_replace(
            F.regexp_replace(valor_limpo, r"\.", ""),
            ",",
            "."
        )
    )
    .when(
        tem_virgula & tem_ponto & (~virgula_depois_ponto),
        F.regexp_replace(valor_limpo, ",", "")
    )
    .when(
        tem_virgula,
        F.regexp_replace(valor_limpo, ",", ".")
    )
    .otherwise(valor_limpo)
)
df_custos = (
    df_custos
    .withColumn(
        "_custo_normalizado",
        valor_normalizado
    )
    .withColumn(
        "custo",
        F.col("_custo_normalizado").cast("decimal(12,2)")
    )
    .withColumn(
        "custo_valido",
        F.col("custo").isNotNull() &
        (F.col("custo") >= F.lit(0))
    )
    .drop("_custo_normalizado")
    .dropDuplicates()
)
(
    df_custos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.custos")
)

display(df_custos.limit(10))

### 2.3. Canais
**Tratamentos realizados: remoção de espaços, padronização dos textos em letras minúsculas, validação dos campos obrigatórios e remoção de registros duplicados exatos.**

In [0]:
df_canais_bronze = spark.table(
    f"{bronze_schema}.tb_canais"
)

df_canais = (
    df_canais_bronze
    .withColumn(
        "id_canal",
        F.trim(F.col("id_canal").cast("string"))
    )
    .withColumn(
        "nome_canal",
        F.lower(F.trim(F.col("nome_canal").cast("string")))
    )
    .withColumn(
        "tipo_canal",
        F.lower(F.trim(F.col("tipo_canal").cast("string")))
    )
    .withColumn(
        "status_canal",
        F.lower(F.trim(F.col("status_canal").cast("string")))
    )
    .withColumn(
        "canal_valido",
        F.col("id_canal").isNotNull() &
        (F.col("id_canal") != "") &
        F.col("nome_canal").isNotNull() &
        (F.col("nome_canal") != "") &
        F.col("tipo_canal").isNotNull() &
        (F.col("tipo_canal") != "") &
        F.col("status_canal").isNotNull() &
        (F.col("status_canal") != "")
    )
    .withColumn("nome_canal",
            F.when(F.col("nome_canal").isin("ura", "u r a"), "URA")
            .when(F.col("nome_canal").isin("chatbot", "chat bot"), "Chatbot")
            .when(F.col("nome_canal").isin("atendimento inicial", "atend inicial"), "Atendimento Inicial")
            .when(F.col("nome_canal").isin("atendimento especializado", "atend especializado"), "Atendimento Especializado")
            .otherwise("Não Informado")
    )
    .dropDuplicates()
)

(
    df_canais.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{silver_schema}.canais")
)

display(df_canais)

###3. Relatório de Qualidade

In [0]:
def gerar_resumo(nome_tabela, df, chave, coluna_valida=None):
    total = df.count()

    chaves_nulas = df.filter(
        F.col(chave).isNull() |
        (F.trim(F.col(chave).cast("string")) == "")
    ).count()

    resultado = {
        "tabela": nome_tabela,
        "total_registros": total,
        "chaves_nulas": chaves_nulas
    }

    if coluna_valida:
        invalidos = df.filter(
            ~F.coalesce(F.col(coluna_valida), F.lit(False))
        ).count()
        resultado["registros_invalidos"] = invalidos
    return resultado


df_silver_pesquisa = spark.table(f"{silver_schema}.pesquisa_satisfacao")
df_silver_custos = spark.table(f"{silver_schema}.custos")

resumo = [
    gerar_resumo(
        "pesquisa_satisfacao",
        df_silver_pesquisa,
        "id_pesquisa",
        "nota_valida"
    ),
    gerar_resumo(
        "custos",
        df_silver_custos,
        "id_custo",
        "custo_valido"
    )
]

display(spark.createDataFrame(resumo))